# Taxonomic food-web architecture and null models

Reproducible analysis for temporally resolved taxonomic food webs. The notebook follows the primary taxonomic analysis first and places the cumulative network at the end as a supplementary descriptive analysis.

In [ ]:
# Shared configuration
from pathlib import Path
import os

# Directory containing the audited input CSV files.
# Set FOODWEB_DATA_DIR to use a different location without editing the notebook.
DATA_DIR = Path(os.environ.get("FOODWEB_DATA_DIR", ".")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data/output directory: {DATA_DIR}")


## 1. Construct and audit temporally resolved taxonomic food webs

Build the 16 sampling-specific directed binary food webs from the audited occurrence and trophic-interaction tables. The audit retains observed isolates and verifies network construction before metric calculation.

In [ ]:

from pathlib import Path
import pandas as pd
import networkx as nx

# ============================================================
# TEMPORAL STRUCTURAL FOOD WEBS — STAGE 1: CONSTRUCTION + AUDIT
# Direction convention: resource -> consumer
# ============================================================

NODES_FILE = DATA_DIR / "FW_lagoon_nodes_AUDITED.csv"
LINKS_FILE = DATA_DIR / "FW_lagoon_trophic_links_AUDITED.csv"
SAMPLING_FILE = DATA_DIR / "FW_lagoon_food_web_by_sampling_AUDITED.csv"

# These are the 16 sampling events with macroinvertebrate sampling.
# We define them explicitly so the analysis does not depend on column position.
TEMPORAL_DATES = [
    "14/05/2016", "09/07/2016", "10/09/2016", "20/01/2017",
    "16/03/2017", "06/04/2017", "29/06/2017", "08/09/2017",
    "15/11/2017", "23/12/2017", "23/02/2018", "19/05/2018",
    "11/08/2018", "20/10/2018", "09/03/2019", "01/06/2019",
]

def load_inputs():
    nodes = pd.read_csv(NODES_FILE)
    links = pd.read_csv(LINKS_FILE)
    sampling = pd.read_csv(SAMPLING_FILE)

    nodes["node"] = nodes["node"].astype(str).str.strip()
    links["consumer"] = links["consumer"].astype(str).str.strip()
    links["resource"] = links["resource"].astype(str).str.strip()

    return nodes, links, sampling


def validate_inputs(nodes, links, sampling):
    node_set = set(nodes["node"])

    missing_dates = [d for d in TEMPORAL_DATES if d not in sampling.columns]
    if missing_dates:
        raise ValueError(f"Temporal dates missing from sampling matrix: {missing_dates}")

    duplicated_nodes = nodes.loc[nodes["node"].duplicated(), "node"].tolist()
    if duplicated_nodes:
        raise ValueError(f"Duplicated node names in node table: {duplicated_nodes}")

    link_nodes = set(links["resource"]) | set(links["consumer"])
    unknown_link_nodes = sorted(link_nodes - node_set)
    if unknown_link_nodes:
        raise ValueError(f"Nodes in trophic links absent from node table: {unknown_link_nodes}")

    # Check all taxa appearing in the 16 selected dates.
    observed_taxa = set()
    for date in TEMPORAL_DATES:
        observed_taxa.update(
            sampling[date].dropna().astype(str).str.strip().loc[lambda x: x.ne("")]
        )
    unknown_sampling_nodes = sorted(observed_taxa - node_set)
    if unknown_sampling_nodes:
        raise ValueError(
            f"Taxa in temporal sampling matrix absent from node table: {unknown_sampling_nodes}"
        )

    print("INPUT AUDIT PASSED")
    print(f"Master nodes: {len(nodes)}")
    print(f"Master trophic links: {len(links)}")
    print(f"Selected temporal dates: {len(TEMPORAL_DATES)}")


def taxa_present_on_date(sampling, date):
    taxa = (
        sampling[date]
        .dropna()
        .astype(str)
        .str.strip()
    )
    taxa = taxa[taxa.ne("")]
    # Preserve first occurrence, but flag duplicates separately in the audit.
    return list(dict.fromkeys(taxa.tolist()))


def build_temporal_graph(date, nodes, links, sampling):
    present = taxa_present_on_date(sampling, date)
    present_set = set(present)

    G = nx.DiGraph(date=date)

    # Add ALL empirically observed taxa first, including potential isolates.
    node_meta = nodes.set_index("node").to_dict(orient="index")
    for node in present:
        G.add_node(node, **node_meta[node])

    # Realized trophic links = master links whose two endpoints co-occur on this date.
    realized = links[
        links["resource"].isin(present_set) &
        links["consumer"].isin(present_set)
    ].copy()

    for row in realized.itertuples(index=False):
        G.add_edge(row.resource, row.consumer)

    return G, realized


def audit_temporal_graphs(nodes, links, sampling):
    audit_rows = []
    isolate_rows = []
    node_rows = []
    edge_rows = []

    for date in TEMPORAL_DATES:
        raw = (
            sampling[date].dropna().astype(str).str.strip()
        )
        raw = raw[raw.ne("")]
        duplicated_in_sampling = sorted(raw[raw.duplicated()].unique().tolist())

        G, realized = build_temporal_graph(date, nodes, links, sampling)

        isolates = sorted(nx.isolates(G))
        weak_components = nx.number_weakly_connected_components(G) if len(G) else 0
        strong_components = nx.number_strongly_connected_components(G) if len(G) else 0

        audit_rows.append({
            "date": date,
            "S_observed_nodes": G.number_of_nodes(),
            "L_realized_links": G.number_of_edges(),
            "n_isolates": len(isolates),
            "prop_isolates": len(isolates) / G.number_of_nodes() if G.number_of_nodes() else float("nan"),
            "weak_components": weak_components,
            "strong_components": strong_components,
            "is_DAG": nx.is_directed_acyclic_graph(G),
            "n_indegree_zero": sum(1 for n in G if G.in_degree(n) == 0),
            "n_outdegree_zero": sum(1 for n in G if G.out_degree(n) == 0),
            "duplicated_taxa_in_sampling_n": len(duplicated_in_sampling),
            "duplicated_taxa_in_sampling": "; ".join(duplicated_in_sampling),
        })

        for node in G.nodes:
            node_rows.append({
                "date": date,
                "node": node,
                "in_degree": G.in_degree(node),
                "out_degree": G.out_degree(node),
                "is_isolate": node in isolates,
                "category": G.nodes[node].get("category", ""),
                "FunctionalGroup": G.nodes[node].get("FunctionalGroup", ""),
            })

        for node in isolates:
            isolate_rows.append({
                "date": date,
                "node": node,
                "category": G.nodes[node].get("category", ""),
                "FunctionalGroup": G.nodes[node].get("FunctionalGroup", ""),
            })

        for row in realized.itertuples(index=False):
            edge_rows.append({
                "date": date,
                "resource": row.resource,
                "consumer": row.consumer,
            })

    audit = pd.DataFrame(audit_rows)
    isolates = pd.DataFrame(isolate_rows)
    temporal_nodes = pd.DataFrame(node_rows)
    temporal_edges = pd.DataFrame(edge_rows)

    return audit, isolates, temporal_nodes, temporal_edges


def main():
    nodes, links, sampling = load_inputs()
    validate_inputs(nodes, links, sampling)

    audit, isolates, temporal_nodes, temporal_edges = audit_temporal_graphs(
        nodes, links, sampling
    )

    audit.to_csv("temporal_structural_network_audit.csv", index=False)
    isolates.to_csv("temporal_structural_isolates.csv", index=False)
    temporal_nodes.to_csv("temporal_structural_nodes_long.csv", index=False)
    temporal_edges.to_csv("temporal_structural_edges_long.csv", index=False)

    print("\nTEMPORAL NETWORK AUDIT")
    print(audit.to_string(index=False))

    print("\nISOLATED NODES")
    if isolates.empty:
        print("No isolated nodes in the 16 temporal networks.")
    else:
        print(isolates.to_string(index=False))

    print("\nSaved:")
    print(" - temporal_structural_network_audit.csv")
    print(" - temporal_structural_isolates.csv")
    print(" - temporal_structural_nodes_long.csv")
    print(" - temporal_structural_edges_long.csv")




In [ ]:
main()


## 2. Calculate observed taxonomic network metrics

Calculate the observed metrics used to describe temporal taxonomic food-web architecture. Directed Infomap is used for community detection and directed modularity is evaluated for the resulting partition.

In [ ]:
%pip -q install infomap


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx

# ============================================================
# TEMPORAL STRUCTURAL FOOD WEBS — STAGE 2: OBSERVED METRICS
# Exact metric definitions from accumulated structural v3.1
# Direction: resource -> consumer
# ============================================================

SEED = 12345
INFOMAP_TRIALS = 20

NODES_FILE = DATA_DIR / "FW_lagoon_nodes_AUDITED.csv"
LINKS_FILE = DATA_DIR / "FW_lagoon_trophic_links_AUDITED.csv"
SAMPLING_FILE = DATA_DIR / "FW_lagoon_food_web_by_sampling_AUDITED.csv"

TEMPORAL_DATES = [
    "14/05/2016", "09/07/2016", "10/09/2016", "20/01/2017",
    "16/03/2017", "06/04/2017", "29/06/2017", "08/09/2017",
    "15/11/2017", "23/12/2017", "23/02/2018", "19/05/2018",
    "11/08/2018", "20/10/2018", "09/03/2019", "01/06/2019",
]

def load_inputs():
    nodes = pd.read_csv(NODES_FILE)
    links = pd.read_csv(LINKS_FILE)
    sampling = pd.read_csv(SAMPLING_FILE)
    nodes["node"] = nodes["node"].astype(str).str.strip()
    links["consumer"] = links["consumer"].astype(str).str.strip()
    links["resource"] = links["resource"].astype(str).str.strip()
    return nodes, links, sampling

def validate_inputs(nodes, links, sampling):
    node_set = set(nodes["node"])
    missing_dates = [d for d in TEMPORAL_DATES if d not in sampling.columns]
    if missing_dates:
        raise ValueError(f"Temporal dates missing from sampling matrix: {missing_dates}")
    if nodes["node"].duplicated().any():
        raise ValueError("Duplicated node names in audited node table.")
    unknown = (set(links["resource"]) | set(links["consumer"])) - node_set
    if unknown:
        raise ValueError(f"Unknown nodes in trophic links: {sorted(unknown)}")
    print("INPUT AUDIT PASSED")
    print(f"Master nodes: {len(nodes)}")
    print(f"Master trophic links: {len(links)}")
    print(f"Selected temporal dates: {len(TEMPORAL_DATES)}")

def taxa_present_on_date(sampling, date):
    x = sampling[date].dropna().astype(str).str.strip()
    x = x[x.ne("")]
    return list(dict.fromkeys(x.tolist()))

def build_temporal_graph(date, nodes, links, sampling):
    present = taxa_present_on_date(sampling, date)
    present_set = set(present)
    G = nx.DiGraph(date=date)
    meta = nodes.set_index("node").to_dict(orient="index")
    for node in present:
        G.add_node(node, **meta[node])
    realized = links[
        links["resource"].isin(present_set) &
        links["consumer"].isin(present_set)
    ].copy()
    G.add_edges_from(realized[["resource", "consumer"]].itertuples(index=False, name=None))
    return G, realized


# ---- Exact v3.1 metric functions ----
def prey_averaged_trophic_levels(G):
    """Prey-averaged TL for DAGs only: basal=1; consumer=1+mean(resource TL).

    Cyclic null networks return NaN rather than forcing a potentially unstable
    linear-system solution. This is deliberate: trophic-level comparisons are
    only made for null models/replicates compatible with this DAG definition.
    """
    if not nx.is_directed_acyclic_graph(G):
        return {n: np.nan for n in G}
    tl = {}
    for n in nx.topological_sort(G):
        prey = list(G.predecessors(n))
        tl[n] = 1.0 if not prey else 1.0 + float(np.mean([tl[p] for p in prey]))
    return tl


def omnivory_index(G, tl):
    out = {}
    for n in G:
        prey = list(G.predecessors(n))
        vals = [tl[p] for p in prey if np.isfinite(tl.get(p, np.nan))]
        out[n] = float(np.std(vals, ddof=0)) if len(vals) >= 2 else (0.0 if np.isfinite(tl.get(n, np.nan)) else np.nan)
    return out


def directed_modularity(G, communities):
    """Leicht-Newman directed modularity Q for a supplied partition."""
    m = G.number_of_edges()
    if m == 0 or not communities: return np.nan
    within = sum(1.0 for i, j in G.edges() if communities.get(i) == communities.get(j))
    groups = {}
    for n, c in communities.items(): groups.setdefault(c, []).append(n)
    expected = sum(sum(G.out_degree(n) for n in members) *
                   sum(G.in_degree(n) for n in members) / m
                   for members in groups.values())
    return float((within - expected) / m)


def infomap_partition(G, seed=SEED, num_trials=INFOMAP_TRIALS):
    """Directed Infomap using the same multi-trial optimization depth for observed and null networks.

    Infomap is stochastic. --num-trials asks Infomap to repeat the optimization
    internally and retain the minimum-codelength solution. A fixed seed makes the
    result reproducible while preserving identical treatment of observed and null graphs.
    """
    try:
        from infomap import Infomap
    except ImportError:
        return None, {"n_modules": np.nan, "codelength": np.nan}
    im = Infomap(f"--directed --silent --seed {seed} --num-trials {num_trials}")
    node_to_int = {n: i for i, n in enumerate(G.nodes())}
    int_to_node = {i: n for n, i in node_to_int.items()}
    for u, v in G.edges():
        im.add_link(node_to_int[u], node_to_int[v])
    im.run()
    part = {int_to_node[x.node_id]: x.module_id for x in im.nodes}
    return part, {"n_modules": len(set(part.values())), "codelength": float(im.codelength)}


def observed_module_diagnostics(G, seed=SEED):
    """Return reproducible observed Infomap partition plus module-size/composition tables."""
    part, info = infomap_partition(G, seed=seed)
    if part is None:
        return pd.DataFrame(), pd.DataFrame(), info
    rows = []
    for node, module in part.items():
        rows.append({
            "module": module,
            "node": node,
            "category": G.nodes[node].get("category", ""),
            "n1_role": G.nodes[node].get("n1_role", ""),
            "in_degree": G.in_degree(node),
            "out_degree": G.out_degree(node),
        })
    members = pd.DataFrame(rows).sort_values(["module", "node"])
    sizes = (members.groupby("module", as_index=False)
             .agg(n_nodes=("node", "size"),
                  nodes=("node", lambda x: "; ".join(map(str, x))),
                  categories=("category", lambda x: "; ".join(sorted(set(map(str, x))))),
                  n1_roles=("n1_role", lambda x: "; ".join(sorted(set(map(str, x))))))
             .sort_values(["n_nodes", "module"], ascending=[False, True]))
    return members, sizes, info

def chain_metrics(G):
    if not nx.is_directed_acyclic_graph(G):
        return {"max_chain_edges": np.nan, "mean_chain_edges": np.nan, "n_basal_terminal_paths": np.nan}
    basal = [n for n in G if G.in_degree(n) == 0]
    top = [n for n in G if G.out_degree(n) == 0]
    lengths = []
    for b in basal:
        for t in top:
            if nx.has_path(G, b, t):
                lengths.extend(len(p)-1 for p in nx.all_simple_paths(G, b, t))
    return {"max_chain_edges": max(lengths) if lengths else np.nan,
            "mean_chain_edges": float(np.mean(lengths)) if lengths else np.nan,
            "n_basal_terminal_paths": len(lengths)}


def calculate_metrics(G, include_infomap=False, seed=SEED):
    S, L = G.number_of_nodes(), G.number_of_edges()
    indeg, outdeg, totaldeg = dict(G.in_degree()), dict(G.out_degree()), dict(G.degree())
    basal = [n for n in G if indeg[n] == 0]
    top = [n for n in G if outdeg[n] == 0]
    inter = [n for n in G if indeg[n] > 0 and outdeg[n] > 0]
    tl = prey_averaged_trophic_levels(G); omni = omnivory_index(G, tl)
    finite_tl = [v for v in tl.values() if np.isfinite(v)]
    finite_oi = [v for v in omni.values() if np.isfinite(v)]
    bet = nx.betweenness_centrality(G, normalized=True)
    close = nx.closeness_centrality(G)
    clustering = nx.clustering(G)
    try: assort = nx.degree_assortativity_coefficient(G, x="out", y="in")
    except Exception: assort = np.nan
    part, info = infomap_partition(G, seed) if include_infomap else (None, {"n_modules":np.nan,"codelength":np.nan})
    gm = {
        "S":S, "L":L, "connectance":L/(S*(S-1)) if S>1 else np.nan,
        "linkage_density_L_per_S":L/S if S else np.nan,
        "mean_total_degree":float(np.mean(list(totaldeg.values()))),
        "mean_in_degree":float(np.mean(list(indeg.values()))), "mean_out_degree":float(np.mean(list(outdeg.values()))),
        "n_basal":len(basal), "prop_basal":len(basal)/S, "n_intermediate":len(inter), "prop_intermediate":len(inter)/S,
        "n_top":len(top), "prop_top":len(top)/S, "n_isolates":nx.number_of_isolates(G),
        "weak_components":nx.number_weakly_connected_components(G), "strong_components":nx.number_strongly_connected_components(G),
        "is_DAG":nx.is_directed_acyclic_graph(G),
        "mean_trophic_level":float(np.mean(finite_tl)) if finite_tl else np.nan,
        "median_trophic_level":float(np.median(finite_tl)) if finite_tl else np.nan,
        "max_trophic_level":float(max(finite_tl)) if finite_tl else np.nan,
        "mean_omnivory_index":float(np.mean(finite_oi)) if finite_oi else np.nan,
        "prop_omnivores_OI_gt_0":float(np.mean([v>0 for v in finite_oi])) if finite_oi else np.nan,
        "mean_betweenness":float(np.mean(list(bet.values()))), "mean_closeness":float(np.mean(list(close.values()))),
        "mean_directed_clustering":float(np.mean(list(clustering.values()))),
        "undirected_projection_transitivity":nx.transitivity(G.to_undirected()) if S>2 else np.nan,
        "out_in_degree_assortativity":assort,
        "infomap_n_modules":info["n_modules"], "infomap_codelength":info["codelength"],
        "directed_modularity_Q_of_infomap_partition":directed_modularity(G, part) if part else np.nan,
    }
    gm.update(chain_metrics(G))
    return gm


def main():
    nodes, links, sampling = load_inputs()
    validate_inputs(nodes, links, sampling)

    metric_rows, audit_rows, isolate_rows = [], [], []

    for i, date in enumerate(TEMPORAL_DATES):
        G, realized = build_temporal_graph(date, nodes, links, sampling)

        # Exactly the same metric function as v3.1 accumulated network.
        metrics = calculate_metrics(G, include_infomap=True, seed=SEED + i)
        metric_rows.append({"date": date, **metrics})

        isolates = sorted(nx.isolates(G))
        audit_rows.append({
            "date": date,
            "S_observed_nodes": G.number_of_nodes(),
            "L_realized_links": G.number_of_edges(),
            "n_isolates": len(isolates),
            "prop_isolates": len(isolates)/G.number_of_nodes() if len(G) else np.nan,
            "weak_components": nx.number_weakly_connected_components(G) if len(G) else 0,
            "strong_components": nx.number_strongly_connected_components(G) if len(G) else 0,
            "is_DAG": nx.is_directed_acyclic_graph(G),
            "n_indegree_zero": sum(G.in_degree(n) == 0 for n in G),
            "n_outdegree_zero": sum(G.out_degree(n) == 0 for n in G),
        })
        for node in isolates:
            isolate_rows.append({
                "date": date, "node": node,
                "category": G.nodes[node].get("category", ""),
                "FunctionalGroup": G.nodes[node].get("FunctionalGroup", "")
            })

    metrics_df = pd.DataFrame(metric_rows)
    audit_df = pd.DataFrame(audit_rows)
    isolates_df = pd.DataFrame(isolate_rows)

    # Independent cross-check against the audit layer.
    check = metrics_df.merge(audit_df, on="date", suffixes=("_metrics", "_audit"))
    pairs = [
        ("S", "S_observed_nodes"), ("L", "L_realized_links"),
        ("n_isolates_metrics", "n_isolates_audit"),
        ("weak_components_metrics", "weak_components_audit"),
        ("strong_components_metrics", "strong_components_audit"),
        ("is_DAG_metrics", "is_DAG_audit"),
    ]
    for a, b in pairs:
        if not (check[a] == check[b]).all():
            raise AssertionError(f"Audit cross-check failed: {a} != {b}")

    metrics_df.to_csv("temporal_structural_observed_metrics.csv", index=False)
    audit_df.to_csv("temporal_structural_observed_metrics_audit.csv", index=False)
    isolates_df.to_csv("temporal_structural_observed_metrics_isolates.csv", index=False)

    print("\nTEMPORAL STRUCTURAL OBSERVED METRICS")
    print(metrics_df.to_string(index=False))
    print("\nAUDIT CROSS-CHECK PASSED FOR ALL 16 NETWORKS.")
    print("\nSaved:")
    print(" - temporal_structural_observed_metrics.csv")
    print(" - temporal_structural_observed_metrics_audit.csv")
    print(" - temporal_structural_observed_metrics_isolates.csv")



In [ ]:
main()


## 3. Evaluate the taxonomic null-model hierarchy

Evaluate each temporal network against the N0–N3 hierarchy, from size/link constraints to exact in- and out-degree preservation. Null-model feasibility and mixing diagnostics are retained as reproducibility checks.

In [ ]:
%pip -q install infomap


In [ ]:
"""Structural food-web metrics and graded null-model analysis — v3.1.

Directed convention: resource -> consumer.
Inputs: audited node and trophic-link tables.

Null gradient
-------------
N0: same S and L only (topological randomization).
N1: same S and L + broad trophic-role composition and minimal role compatibility.
    Broad role labels are randomly reassigned among nodes each replicate while
    preserving their observed counts. Individual node degrees are NOT preserved.
N2: same S and L + each node's observed ecological category + an a-priori
    category-level trophic compatibility matrix. Individual degrees are NOT preserved.
N3: same S and L + exact in- and out-degree sequence via directed double-edge swaps.

The gradient is a set of increasingly informative constraints; it is NOT assumed
that edge overlap or every metric must change monotonically across N0-N3.

Methodological anchors
----------------------
Williams & Martinez (2004), American Naturalist 163:458-468.
  DOI: 10.1086/381964
Rosvall & Bergstrom (2008), PNAS 105:1118-1123.
  DOI: 10.1073/pnas.0706851105
Leicht & Newman (2008), Physical Review Letters 100:118703.
  DOI: 10.1103/PhysRevLett.100.118703

Important interpretation rule
-----------------------------
A null model asks what structure remains exceptional after specific information
is held fixed. N1/N2 rules are ecological constraints, not fitted probabilities.
Observed links are used only to audit that the compatibility matrix does not
accidentally forbid known interactions; absence of an observed category pair is
NOT, by itself, treated as evidence that the pair is biologically impossible.
"""
from __future__ import annotations
import random
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx

NODE_FILE = DATA_DIR / "FW_lagoon_nodes_AUDITED.csv"
LINK_FILE = DATA_DIR / "FW_lagoon_trophic_links_AUDITED.csv"
EXPECTED_NODES = 75
EXPECTED_LINKS = 394
N_NULL = 1000
N_COMMUNITY_NULL = 300  # Infomap is stochastic/expensive; kept explicit and separate.
SEED = 20260926
N3_MIXING_FACTORS = (1, 5, 10, 20, 50)
N3_MIXING_REPLICATES = 50
ZERO_SD_TOL = 1e-12
INFOMAP_TRIALS = 20  # Same optimization depth for observed and null networks.

# Broad N1 roles. These derive from ecological categories, not observed node degrees.
CATEGORY_TO_N1_ROLE = {
    "autotrophic phytoplankton": "autotroph",
    "periphyton": "autotroph",
    "facultative mixotrophic phytoplankton": "mixotroph",
    "mixotrophic phytoplankton": "mixotroph",
    "DOM": "detrital_resource", "FPOM": "detrital_resource", "CPOM": "detrital_resource",
    "microbial heterotroph": "microbial_consumer",
    "consumer": "zooplankton_consumer",
    "scraper": "grazer_collector", "filtering collector": "grazer_collector",
    "gathering collector": "grazer_collector",
    "predator": "predator_omnivore", "omnivorous": "predator_omnivore",
}

# Minimal broad trophic architecture for N1. This is deliberately permissive.
N1_ALLOWED_ROLE_PAIRS = {
    ("autotroph", "zooplankton_consumer"),
    ("autotroph", "grazer_collector"),
    ("autotroph", "predator_omnivore"),
    ("mixotroph", "zooplankton_consumer"),
    ("mixotroph", "grazer_collector"),
    ("mixotroph", "predator_omnivore"),
    ("detrital_resource", "mixotroph"),
    ("detrital_resource", "microbial_consumer"),
    ("detrital_resource", "zooplankton_consumer"),
    ("detrital_resource", "grazer_collector"),
    ("detrital_resource", "predator_omnivore"),
    ("microbial_consumer", "zooplankton_consumer"),
    ("microbial_consumer", "grazer_collector"),
    ("microbial_consumer", "predator_omnivore"),
    ("zooplankton_consumer", "grazer_collector"),
    ("zooplankton_consumer", "predator_omnivore"),
    ("grazer_collector", "grazer_collector"),
    ("grazer_collector", "predator_omnivore"),
    ("predator_omnivore", "predator_omnivore"),
}

# N2: category-level potential trophic compatibility. Rules are intentionally
# broader than the realized 394 links and are based on guild meaning. They do
# not preserve observed degrees or observed category-pair frequencies.
N2_ALLOWED_CATEGORY_PAIRS = {
    # Mixotrophic/microbial uptake represented in this dataset.
    ("DOM", "facultative mixotrophic phytoplankton"),
    ("DOM", "mixotrophic phytoplankton"),
    ("DOM", "microbial heterotroph"),

    # Zooplankton consumer: planktonic producers/mixotrophs, microbial pathway,
    # fine detritus and periphyton are left admissible at this coarse resolution.
    *( (r, "consumer") for r in [
        "autotrophic phytoplankton", "facultative mixotrophic phytoplankton",
        "mixotrophic phytoplankton", "microbial heterotroph", "FPOM", "periphyton"] ),

    # Filtering collectors: suspended phytoplankton/mixotrophs, microbes and FPOM.
    *( (r, "filtering collector") for r in [
        "autotrophic phytoplankton", "facultative mixotrophic phytoplankton",
        "mixotrophic phytoplankton", "microbial heterotroph", "FPOM"] ),

    # Gathering collectors: deposited detritus/periphyton/microbes and small
    # benthic consumers; broad guild labels are not interpreted as exclusive diets.
    *( (r, "gathering collector") for r in [
        "FPOM", "CPOM", "periphyton", "microbial heterotroph",
        "gathering collector"] ),

    # Scrapers: periphyton/algae plus detrital material; category is broad enough
    # to admit documented animal prey for Corixidae in the audited web.
    *( (r, "scraper") for r in [
        "periphyton", "autotrophic phytoplankton", "facultative mixotrophic phytoplankton",
        "mixotrophic phytoplankton", "FPOM", "CPOM", "consumer", "gathering collector"] ),

    # Omnivores: broad plant/detrital/microbial and animal resources.
    *( (r, "omnivorous") for r in [
        "periphyton", "autotrophic phytoplankton", "facultative mixotrophic phytoplankton",
        "mixotrophic phytoplankton", "FPOM", "CPOM", "microbial heterotroph",
        "consumer", "scraper", "filtering collector", "gathering collector"] ),

    # Predators: animal prey, including intraguild predation at this taxonomic resolution.
    *( (r, "predator") for r in [
        "consumer", "scraper", "filtering collector", "gathering collector",
        "omnivorous", "predator"] ),
}


def load_network(node_file=NODE_FILE, link_file=LINK_FILE):
    nodes = pd.read_csv(node_file)
    links = pd.read_csv(link_file)
    if "node" not in nodes or not {"resource", "consumer"}.issubset(links.columns):
        raise ValueError("Required columns are missing.")
    if nodes["node"].isna().any() or links[["resource", "consumer"]].isna().any().any():
        raise ValueError("Missing node names or trophic-link endpoints.")
    if nodes["node"].duplicated().any() or links[["resource", "consumer"]].duplicated().any():
        raise ValueError("Duplicated nodes or trophic links.")
    node_set = set(nodes["node"])
    missing = (set(links["resource"]) | set(links["consumer"])) - node_set
    if missing: raise ValueError(f"Link endpoints absent from node table: {sorted(missing)}")
    if (links["resource"] == links["consumer"]).any():
        raise ValueError("Self-loops found; current analysis assumes none.")
    if "category" not in nodes.columns:
        raise ValueError("N1/N2 require the audited 'category' column.")
    unknown = set(nodes["category"].dropna()) - set(CATEGORY_TO_N1_ROLE)
    if unknown: raise ValueError(f"Categories missing from N1 role map: {sorted(unknown)}")

    G = nx.DiGraph()
    meta_cols = [c for c in nodes.columns if c != "node"]
    for _, row in nodes.iterrows():
        attrs = {c: row[c] for c in meta_cols}
        attrs["n1_role"] = CATEGORY_TO_N1_ROLE[row["category"]]
        G.add_node(row["node"], **attrs)
    G.add_edges_from(links[["resource", "consumer"]].itertuples(index=False, name=None))
    return nodes, links, G


def prey_averaged_trophic_levels(G):
    """Prey-averaged TL for DAGs only: basal=1; consumer=1+mean(resource TL).

    Cyclic null networks return NaN rather than forcing a potentially unstable
    linear-system solution. This is deliberate: trophic-level comparisons are
    only made for null models/replicates compatible with this DAG definition.
    """
    if not nx.is_directed_acyclic_graph(G):
        return {n: np.nan for n in G}
    tl = {}
    for n in nx.topological_sort(G):
        prey = list(G.predecessors(n))
        tl[n] = 1.0 if not prey else 1.0 + float(np.mean([tl[p] for p in prey]))
    return tl


def omnivory_index(G, tl):
    out = {}
    for n in G:
        prey = list(G.predecessors(n))
        vals = [tl[p] for p in prey if np.isfinite(tl.get(p, np.nan))]
        out[n] = float(np.std(vals, ddof=0)) if len(vals) >= 2 else (0.0 if np.isfinite(tl.get(n, np.nan)) else np.nan)
    return out


def directed_modularity(G, communities):
    """Leicht-Newman directed modularity Q for a supplied partition."""
    m = G.number_of_edges()
    if m == 0 or not communities: return np.nan
    within = sum(1.0 for i, j in G.edges() if communities.get(i) == communities.get(j))
    groups = {}
    for n, c in communities.items(): groups.setdefault(c, []).append(n)
    expected = sum(sum(G.out_degree(n) for n in members) *
                   sum(G.in_degree(n) for n in members) / m
                   for members in groups.values())
    return float((within - expected) / m)


def infomap_partition(G, seed=SEED, num_trials=INFOMAP_TRIALS):
    """Directed Infomap using the same multi-trial optimization depth for observed and null networks.

    Infomap is stochastic. --num-trials asks Infomap to repeat the optimization
    internally and retain the minimum-codelength solution. A fixed seed makes the
    result reproducible while preserving identical treatment of observed and null graphs.
    """
    try:
        from infomap import Infomap
    except ImportError:
        return None, {"n_modules": np.nan, "codelength": np.nan}
    im = Infomap(f"--directed --silent --seed {seed} --num-trials {num_trials}")
    node_to_int = {n: i for i, n in enumerate(G.nodes())}
    int_to_node = {i: n for n, i in node_to_int.items()}
    for u, v in G.edges():
        im.add_link(node_to_int[u], node_to_int[v])
    im.run()
    part = {int_to_node[x.node_id]: x.module_id for x in im.nodes}
    return part, {"n_modules": len(set(part.values())), "codelength": float(im.codelength)}


def observed_module_diagnostics(G, seed=SEED):
    """Return reproducible observed Infomap partition plus module-size/composition tables."""
    part, info = infomap_partition(G, seed=seed)
    if part is None:
        return pd.DataFrame(), pd.DataFrame(), info
    rows = []
    for node, module in part.items():
        rows.append({
            "module": module,
            "node": node,
            "category": G.nodes[node].get("category", ""),
            "n1_role": G.nodes[node].get("n1_role", ""),
            "in_degree": G.in_degree(node),
            "out_degree": G.out_degree(node),
        })
    members = pd.DataFrame(rows).sort_values(["module", "node"])
    sizes = (members.groupby("module", as_index=False)
             .agg(n_nodes=("node", "size"),
                  nodes=("node", lambda x: "; ".join(map(str, x))),
                  categories=("category", lambda x: "; ".join(sorted(set(map(str, x))))),
                  n1_roles=("n1_role", lambda x: "; ".join(sorted(set(map(str, x))))))
             .sort_values(["n_nodes", "module"], ascending=[False, True]))
    return members, sizes, info

def chain_metrics(G):
    if not nx.is_directed_acyclic_graph(G):
        return {"max_chain_edges": np.nan, "mean_chain_edges": np.nan, "n_basal_terminal_paths": np.nan}
    basal = [n for n in G if G.in_degree(n) == 0]
    top = [n for n in G if G.out_degree(n) == 0]
    lengths = []
    for b in basal:
        for t in top:
            if nx.has_path(G, b, t):
                lengths.extend(len(p)-1 for p in nx.all_simple_paths(G, b, t))
    return {"max_chain_edges": max(lengths) if lengths else np.nan,
            "mean_chain_edges": float(np.mean(lengths)) if lengths else np.nan,
            "n_basal_terminal_paths": len(lengths)}


def calculate_metrics(G, include_infomap=False, seed=SEED):
    S, L = G.number_of_nodes(), G.number_of_edges()
    indeg, outdeg, totaldeg = dict(G.in_degree()), dict(G.out_degree()), dict(G.degree())
    basal = [n for n in G if indeg[n] == 0]
    top = [n for n in G if outdeg[n] == 0]
    inter = [n for n in G if indeg[n] > 0 and outdeg[n] > 0]
    tl = prey_averaged_trophic_levels(G); omni = omnivory_index(G, tl)
    finite_tl = [v for v in tl.values() if np.isfinite(v)]
    finite_oi = [v for v in omni.values() if np.isfinite(v)]
    bet = nx.betweenness_centrality(G, normalized=True)
    close = nx.closeness_centrality(G)
    clustering = nx.clustering(G)
    try: assort = nx.degree_assortativity_coefficient(G, x="out", y="in")
    except Exception: assort = np.nan
    part, info = infomap_partition(G, seed) if include_infomap else (None, {"n_modules":np.nan,"codelength":np.nan})
    gm = {
        "S":S, "L":L, "connectance":L/(S*(S-1)) if S>1 else np.nan,
        "linkage_density_L_per_S":L/S if S else np.nan,
        "mean_total_degree":float(np.mean(list(totaldeg.values()))),
        "mean_in_degree":float(np.mean(list(indeg.values()))), "mean_out_degree":float(np.mean(list(outdeg.values()))),
        "n_basal":len(basal), "prop_basal":len(basal)/S, "n_intermediate":len(inter), "prop_intermediate":len(inter)/S,
        "n_top":len(top), "prop_top":len(top)/S, "n_isolates":nx.number_of_isolates(G),
        "weak_components":nx.number_weakly_connected_components(G), "strong_components":nx.number_strongly_connected_components(G),
        "is_DAG":nx.is_directed_acyclic_graph(G),
        "mean_trophic_level":float(np.mean(finite_tl)) if finite_tl else np.nan,
        "median_trophic_level":float(np.median(finite_tl)) if finite_tl else np.nan,
        "max_trophic_level":float(max(finite_tl)) if finite_tl else np.nan,
        "mean_omnivory_index":float(np.mean(finite_oi)) if finite_oi else np.nan,
        "prop_omnivores_OI_gt_0":float(np.mean([v>0 for v in finite_oi])) if finite_oi else np.nan,
        "mean_betweenness":float(np.mean(list(bet.values()))), "mean_closeness":float(np.mean(list(close.values()))),
        "mean_directed_clustering":float(np.mean(list(clustering.values()))),
        "undirected_projection_transitivity":nx.transitivity(G.to_undirected()) if S>2 else np.nan,
        "out_in_degree_assortativity":assort,
        "infomap_n_modules":info["n_modules"], "infomap_codelength":info["codelength"],
        "directed_modularity_Q_of_infomap_partition":directed_modularity(G, part) if part else np.nan,
    }
    gm.update(chain_metrics(G))
    return gm


# ------------------------------ NULL MODELS ------------------------------
def null_N0_same_S_L(G, rng):
    H=nx.DiGraph(); H.add_nodes_from(G.nodes(data=True))
    possible=[(u,v) for u in G for v in G if u!=v]
    H.add_edges_from(rng.sample(possible, G.number_of_edges())); return H


def null_N1_broad_roles(G, rng):
    """N1: preserve broad-role counts, not role identity of labelled nodes."""
    H=nx.DiGraph(); H.add_nodes_from(G.nodes(data=True))
    roles=[G.nodes[n]["n1_role"] for n in G]
    rng.shuffle(roles)
    assigned=dict(zip(G.nodes(), roles))
    nx.set_node_attributes(H, assigned, "n1_role_randomized")
    possible=[(u,v) for u in G for v in G if u!=v and (assigned[u],assigned[v]) in N1_ALLOWED_ROLE_PAIRS]
    if len(possible)<G.number_of_edges(): raise ValueError(f"N1 admissible space {len(possible)} < L")
    H.add_edges_from(rng.sample(possible,G.number_of_edges()))
    # Internal audit: role counts and every generated edge must obey N1 by construction.
    original_counts = pd.Series([G.nodes[n]["n1_role"] for n in G]).value_counts().sort_index()
    randomized_counts = pd.Series(list(assigned.values())).value_counts().sort_index()
    if not original_counts.equals(randomized_counts):
        raise AssertionError("N1 role counts changed during randomization.")
    if any((assigned[u], assigned[v]) not in N1_ALLOWED_ROLE_PAIRS for u, v in H.edges()):
        raise AssertionError("N1 generated an inadmissible broad-role edge.")
    return H


def null_N2_ecological_compatibility(G, rng):
    """N2: preserve each node's ecological category and category-level compatibility."""
    possible=[]
    for u in G:
        cu=G.nodes[u]["category"]
        for v in G:
            if u==v: continue
            cv=G.nodes[v]["category"]
            if (cu,cv) in N2_ALLOWED_CATEGORY_PAIRS: possible.append((u,v))
    if len(possible)<G.number_of_edges(): raise ValueError(f"N2 admissible space {len(possible)} < L")
    H=nx.DiGraph(); H.add_nodes_from(G.nodes(data=True)); H.add_edges_from(rng.sample(possible,G.number_of_edges())); return H


def null_N3_degree_preserving(G, rng, nswap_factor=20, max_tries_factor=200):
    H=G.copy(); target=max(1,nswap_factor*G.number_of_edges()); max_tries=max_tries_factor*target
    swaps=tries=0; edges=list(H.edges())
    while swaps<target and tries<max_tries:
        tries+=1; (a,b),(c,d)=rng.sample(edges,2)
        if len({a,b,c,d})<4 or a==d or c==b or H.has_edge(a,d) or H.has_edge(c,b): continue
        H.remove_edge(a,b);H.remove_edge(c,d);H.add_edge(a,d);H.add_edge(c,b);edges=list(H.edges());swaps+=1
    H.graph.update(swaps_achieved=swaps,swap_target=target,swap_tries=tries); return H


def audit_ecological_rules(G):
    """Audit role counts, admissible-space sizes and whether N2 forbids observed category pairs."""
    role_counts=pd.Series([G.nodes[n]["n1_role"] for n in G]).value_counts().rename_axis("n1_role").reset_index(name="n_nodes")
    n1_space=sum(1 for u in G for v in G if u!=v and (G.nodes[u]["n1_role"],G.nodes[v]["n1_role"]) in N1_ALLOWED_ROLE_PAIRS)
    n2_space=sum(1 for u in G for v in G if u!=v and (G.nodes[u]["category"],G.nodes[v]["category"]) in N2_ALLOWED_CATEGORY_PAIRS)
    forbidden=[]
    for u,v in G.edges():
        pair=(G.nodes[u]["category"],G.nodes[v]["category"])
        if pair not in N2_ALLOWED_CATEGORY_PAIRS: forbidden.append((u,v,*pair))
    return role_counts,n1_space,n2_space,pd.DataFrame(forbidden,columns=["resource","consumer","resource_category","consumer_category"])


def run_null_ensemble(G, generator, n=N_NULL, seed=SEED, include_infomap=False, **kwargs):
    rng=random.Random(seed); obs_edges=set(G.edges()); rows=[]
    for r in range(n):
        H=generator(G,rng=rng,**kwargs)
        row=calculate_metrics(H,include_infomap=include_infomap,seed=seed+r+1)
        row.update(replicate=r+1,edge_overlap_fraction=len(obs_edges & set(H.edges()))/G.number_of_edges(),
                   swaps_achieved=H.graph.get("swaps_achieved",np.nan),swap_target=H.graph.get("swap_target",np.nan))
        rows.append(row)
    return pd.DataFrame(rows)


def run_community_ensemble(G, generator, n=N_COMMUNITY_NULL, seed=SEED, **kwargs):
    """Separate Infomap ensemble so community inference has explicit replication."""
    return run_null_ensemble(G,generator,n=n,seed=seed,include_infomap=True,**kwargs)


def n3_mixing_diagnostic(G, factors=N3_MIXING_FACTORS, n_replicates=N3_MIXING_REPLICATES, seed=SEED):
    obs=set(G.edges()); rows=[]; master=random.Random(seed)
    for factor in factors:
        for r in range(n_replicates):
            H=null_N3_degree_preserving(G,random.Random(master.randrange(2**32)),nswap_factor=factor)
            rows.append({"nswap_factor":factor,"replicate":r+1,
                         "edge_overlap_fraction":len(obs & set(H.edges()))/G.number_of_edges(),
                         "swaps_achieved":H.graph["swaps_achieved"],"swap_target":H.graph["swap_target"],
                         "target_fraction_achieved":H.graph["swaps_achieved"]/H.graph["swap_target"]})
    return pd.DataFrame(rows)


def compare_observed_to_null(observed,null_df,null_name):
    rows=[]; skip={"replicate","swaps_achieved","swap_target","edge_overlap_fraction"}
    for metric in null_df.select_dtypes(include=[np.number]).columns:
        if metric in skip or metric not in observed or isinstance(observed[metric],(bool,np.bool_)): continue
        vals=null_df[metric].dropna().to_numpy(float)
        if not len(vals): continue
        obs=float(observed[metric]); mu=float(np.mean(vals)); sd=float(np.std(vals,ddof=1)) if len(vals)>1 else np.nan
        invariant=(not np.isfinite(sd)) or sd<ZERO_SD_TOL
        z=np.nan if invariant else (obs-mu)/sd
        if invariant:
            p=np.nan if abs(obs-mu)>ZERO_SD_TOL else 1.0
        else:
            dev=abs(obs-mu); p=(1+np.sum(np.abs(vals-mu)>=dev))/(len(vals)+1)
        rows.append({"null_model":null_name,"metric":metric,"observed":obs,"n_valid_null":len(vals),
                     "null_mean":mu,"null_sd":sd,"null_q025":float(np.quantile(vals,.025)),"null_q975":float(np.quantile(vals,.975)),
                     "z_score":z,"p_empirical_two_sided":p,"invariant_under_null":invariant})
    return pd.DataFrame(rows)


def summarize_overlap(ensembles):
    rows=[]
    for name,df in ensembles.items():
        x=df["edge_overlap_fraction"].dropna()
        rows.append({"null_model":name,"n":len(x),"overlap_mean":x.mean(),"overlap_sd":x.std(),
                     "overlap_q025":x.quantile(.025),"overlap_q975":x.quantile(.975)})
    return pd.DataFrame(rows)



def build_funnel_summary(summary, community_summary, compatibility, alpha=0.05):
    """Summarize where each metric ceases to differ across the graded null funnel.

    This is a descriptive decision aid, not a biological conclusion. Metrics marked
    not recommended/descriptive-invariant are not used to declare a transition.
    """
    allsum = pd.concat([
        summary.assign(source="general"),
        community_summary.assign(source="community")
    ], ignore_index=True)
    allsum["null_model_base"] = allsum["null_model"].str.replace("_community", "", regex=False)
    order = ["N0_same_S_L", "N1_broad_roles", "N2_ecological_compatibility", "N3_degree_preserving"]
    rows = []
    for metric, g in allsum.groupby("metric"):
        states = {}
        for name in order:
            hit = g[g["null_model_base"] == name]
            if hit.empty:
                states[name] = "not tested"
                continue
            comp = compatibility[(compatibility["null_model"] == name) & (compatibility["metric"] == metric)]
            status = comp["status"].iloc[0] if not comp.empty else "inferential"
            p = hit["p_empirical_two_sided"].iloc[0]
            if status in {"not recommended", "descriptive/invariant"}:
                states[name] = status
            elif pd.isna(p):
                states[name] = "NA"
            else:
                states[name] = "differs" if p < alpha else "not clearly different"
        first_explained = next((n for n in order if states[n] == "not clearly different"), "")
        rows.append({"metric": metric, **states, "first_null_not_clearly_different": first_explained})
    return pd.DataFrame(rows)


# ============================================================
# TEMPORAL STRUCTURAL FOOD WEBS — STAGE 3: N0–N3 NULL MODELS
# Same null definitions and metric engine as accumulated v3.1.
# Each date is its own observed network/universe.
# Direction: resource -> consumer
# ============================================================

SAMPLING_FILE = DATA_DIR / "FW_lagoon_food_web_by_sampling_AUDITED.csv"

TEMPORAL_DATES = [
    "14/05/2016", "09/07/2016", "10/09/2016", "20/01/2017",
    "16/03/2017", "06/04/2017", "29/06/2017", "08/09/2017",
    "15/11/2017", "23/12/2017", "23/02/2018", "19/05/2018",
    "11/08/2018", "20/10/2018", "09/03/2019", "01/06/2019",
]

def load_temporal_inputs():
    nodes = pd.read_csv(NODE_FILE)
    links = pd.read_csv(LINK_FILE)
    sampling = pd.read_csv(SAMPLING_FILE)

    nodes["node"] = nodes["node"].astype(str).str.strip()
    links["resource"] = links["resource"].astype(str).str.strip()
    links["consumer"] = links["consumer"].astype(str).str.strip()

    if len(nodes) != EXPECTED_NODES or len(links) != EXPECTED_LINKS:
        raise ValueError(
            f"Master audit mismatch: expected {EXPECTED_NODES} nodes/{EXPECTED_LINKS} links, "
            f"found {len(nodes)}/{len(links)}."
        )
    if nodes["node"].duplicated().any() or links[["resource","consumer"]].duplicated().any():
        raise ValueError("Duplicated nodes or trophic links in audited master tables.")
    if "category" not in nodes.columns:
        raise ValueError("Audited node table requires 'category'.")
    unknown_categories = set(nodes["category"].dropna()) - set(CATEGORY_TO_N1_ROLE)
    if unknown_categories:
        raise ValueError(f"Categories missing from N1 role map: {sorted(unknown_categories)}")
    missing_dates = [d for d in TEMPORAL_DATES if d not in sampling.columns]
    if missing_dates:
        raise ValueError(f"Temporal dates absent from sampling matrix: {missing_dates}")

    print("INPUT AUDIT PASSED")
    print(f"Master nodes: {len(nodes)}")
    print(f"Master trophic links: {len(links)}")
    print(f"Selected temporal dates: {len(TEMPORAL_DATES)}")
    return nodes, links, sampling


def taxa_present_on_date(sampling, date):
    x = sampling[date].dropna().astype(str).str.strip()
    x = x[x.ne("")]
    return list(dict.fromkeys(x.tolist()))


def build_temporal_graph(date, nodes, links, sampling):
    present = taxa_present_on_date(sampling, date)
    present_set = set(present)
    node_lookup = nodes.set_index("node")

    unknown = sorted(present_set - set(node_lookup.index))
    if unknown:
        raise ValueError(f"{date}: taxa absent from audited node table: {unknown}")

    G = nx.DiGraph(date=date)
    meta_cols = [c for c in nodes.columns if c != "node"]

    # Add every empirically observed taxon FIRST, including isolates.
    for node in present:
        row = node_lookup.loc[node]
        attrs = {c: row[c] for c in meta_cols}
        attrs["n1_role"] = CATEGORY_TO_N1_ROLE[row["category"]]
        G.add_node(node, **attrs)

    # Realized temporal links are induced from the same audited 394-link master web.
    realized = links[
        links["resource"].isin(present_set) &
        links["consumer"].isin(present_set)
    ].copy()
    G.add_edges_from(realized[["resource","consumer"]].itertuples(index=False, name=None))
    return G


def temporal_feasibility_row(date, G):
    roles, n1space, n2space, forbidden = audit_ecological_rules(G)
    S, L = G.number_of_nodes(), G.number_of_edges()
    return {
        "date": date,
        "S": S,
        "L": L,
        "n_isolates": nx.number_of_isolates(G),
        "is_DAG": nx.is_directed_acyclic_graph(G),
        "N0_admissible_pairs": S * (S - 1),
        "N1_admissible_pairs": n1space,
        "N2_admissible_pairs": n2space,
        "N1_capacity_minus_L": n1space - L,
        "N2_capacity_minus_L": n2space - L,
        "N2_forbidden_observed_links": len(forbidden),
    }, roles, forbidden


def main():
    nodes, links, sampling = load_temporal_inputs()

    generators = {
        "N0_same_S_L": (null_N0_same_S_L, {}),
        "N1_broad_roles": (null_N1_broad_roles, {}),
        "N2_ecological_compatibility": (null_N2_ecological_compatibility, {}),
        "N3_degree_preserving": (null_N3_degree_preserving, {"nswap_factor": 20}),
    }

    # --------------------------------------------------------
    # A. Preflight: build all 16 graphs and audit null feasibility
    # --------------------------------------------------------
    graphs = {}
    observed_by_date = {}
    feasibility_rows = []
    role_rows = []
    forbidden_rows = []

    print("\n=== PREFLIGHT: TEMPORAL NULL FEASIBILITY ===")
    for di, date in enumerate(TEMPORAL_DATES):
        G = build_temporal_graph(date, nodes, links, sampling)
        graphs[date] = G

        observed = calculate_metrics(G, include_infomap=True, seed=SEED + di)
        observed_by_date[date] = observed

        row, roles, forbidden = temporal_feasibility_row(date, G)
        feasibility_rows.append(row)

        roles = roles.copy()
        roles.insert(0, "date", date)
        role_rows.append(roles)

        if len(forbidden):
            forbidden = forbidden.copy()
            forbidden.insert(0, "date", date)
            forbidden_rows.append(forbidden)

        if row["N1_admissible_pairs"] < row["L"]:
            raise ValueError(f"{date}: N1 admissible space < observed L.")
        if row["N2_admissible_pairs"] < row["L"]:
            raise ValueError(f"{date}: N2 admissible space < observed L.")
        if row["N2_forbidden_observed_links"] > 0:
            raise ValueError(f"{date}: frozen N2 matrix forbids observed temporal links.")

    feasibility = pd.DataFrame(feasibility_rows)
    roles_all = pd.concat(role_rows, ignore_index=True)
    forbidden_all = (
        pd.concat(forbidden_rows, ignore_index=True)
        if forbidden_rows else
        pd.DataFrame(columns=["date","resource","consumer","resource_category","consumer_category"])
    )

    feasibility.to_csv("temporal_structural_null_feasibility_v31.csv", index=False)
    roles_all.to_csv("temporal_structural_N1_role_counts_v31.csv", index=False)
    forbidden_all.to_csv("temporal_structural_N2_forbidden_observed_links_v31.csv", index=False)

    print(feasibility.to_string(index=False))
    print("\nPreflight passed: N1/N2 are feasible for all dates and N2 forbids no observed temporal link.")

    # Save observed metrics again here so Stage 3 is self-contained and traceable.
    observed_table = pd.DataFrame(
        [{"date": d, **observed_by_date[d]} for d in TEMPORAL_DATES]
    )
    observed_table.to_csv("temporal_structural_observed_metrics_stage3_v31.csv", index=False)

    # --------------------------------------------------------
    # B. N3 mixing diagnostic BEFORE inferential ensembles
    # --------------------------------------------------------
    print("\n=== N3 MIXING DIAGNOSTIC ===")
    mixing_all = []
    mixing_summary_all = []

    for di, date in enumerate(TEMPORAL_DATES):
        G = graphs[date]
        mix = n3_mixing_diagnostic(
            G,
            factors=N3_MIXING_FACTORS,
            n_replicates=N3_MIXING_REPLICATES,
            seed=SEED + 100000 + di * 1000,
        )
        mix.insert(0, "date", date)
        mixing_all.append(mix)

        ms = (
            mix.groupby("nswap_factor", as_index=False)
            .agg(
                overlap_mean=("edge_overlap_fraction","mean"),
                overlap_sd=("edge_overlap_fraction","std"),
                target_fraction_mean=("target_fraction_achieved","mean"),
                swaps_achieved_mean=("swaps_achieved","mean"),
                swap_target_mean=("swap_target","mean"),
            )
        )
        ms.insert(0, "date", date)
        mixing_summary_all.append(ms)
        print(f"\n{date}\n{ms.to_string(index=False)}")

    mixing_all = pd.concat(mixing_all, ignore_index=True)
    mixing_summary_all = pd.concat(mixing_summary_all, ignore_index=True)
    mixing_all.to_csv("temporal_structural_N3_mixing_diagnostic_v31.csv", index=False)
    mixing_summary_all.to_csv("temporal_structural_N3_mixing_summary_v31.csv", index=False)

    # --------------------------------------------------------
    # C. Full metric null ensembles: 1000 replicates/model/date
    #    Infomap is intentionally excluded here; it gets its own
    #    300-replicate community ensemble below, as in cumulative v3.1.
    # --------------------------------------------------------
    print("\n=== FULL TEMPORAL NULL ENSEMBLES ===")
    all_null_rows = []
    all_comparisons = []
    all_overlap_rows = []
    compatibility_rows = []

    trophic = {
        "mean_trophic_level","median_trophic_level","max_trophic_level",
        "mean_omnivory_index","prop_omnivores_OI_gt_0",
        "max_chain_edges","mean_chain_edges","n_basal_terminal_paths"
    }
    fixed_common = {
        "S","L","connectance","linkage_density_L_per_S",
        "mean_total_degree","mean_in_degree","mean_out_degree"
    }

    for di, date in enumerate(TEMPORAL_DATES):
        G = graphs[date]
        observed = observed_by_date[date]
        ensembles = {}

        print(f"\n######## DATE {date} | S={G.number_of_nodes()} L={G.number_of_edges()} ########")

        for mi, (name, (gen, kwargs)) in enumerate(generators.items()):
            model_seed = SEED + 1000000 + di * 100000 + mi * 10000
            print(f"Running {name}: {N_NULL} replicates...")
            df = run_null_ensemble(
                G, gen, n=N_NULL, seed=model_seed,
                include_infomap=False, **kwargs
            )
            df.insert(0, "null_model", name)
            df.insert(0, "date", date)
            all_null_rows.append(df)
            ensembles[name] = df.drop(columns=["date","null_model"])

            comp = compare_observed_to_null(
                observed, ensembles[name], name
            )
            comp.insert(0, "date", date)
            all_comparisons.append(comp)

        ov = summarize_overlap(ensembles)
        ov.insert(0, "date", date)
        all_overlap_rows.append(ov)

        # Metric x null compatibility is date-specific because DAG availability
        # can differ among temporal null ensembles.
        for name in generators:
            edf = ensembles[name]
            for metric in observed:
                status = "inferential"
                reason = ""
                if metric in fixed_common:
                    status = "descriptive/invariant"
                    reason = "fixed by S and L"
                if name == "N3_degree_preserving" and metric in {
                    "n_basal","prop_basal","n_intermediate","prop_intermediate",
                    "n_top","prop_top","n_isolates"
                }:
                    status = "descriptive/invariant"
                    reason = "fixed by exact in/out-degree sequence"
                if metric in trophic:
                    valid = int(edf[metric].notna().sum()) if metric in edf else 0
                    if valid < max(30, int(.5 * N_NULL)):
                        status = "not recommended"
                        reason = f"DAG-compatible value available in only {valid}/{N_NULL} null replicates"
                    elif valid < N_NULL:
                        status = "conditional"
                        reason = f"computed only for {valid}/{N_NULL} DAG-compatible null replicates"
                compatibility_rows.append({
                    "date": date, "null_model": name, "metric": metric,
                    "status": status, "reason": reason
                })

    null_all = pd.concat(all_null_rows, ignore_index=True)
    comparison_all = pd.concat(all_comparisons, ignore_index=True)
    overlap_all = pd.concat(all_overlap_rows, ignore_index=True)
    compatibility_all = pd.DataFrame(compatibility_rows)

    null_all.to_csv("temporal_structural_null_ensembles_all_v31.csv", index=False)
    comparison_all.to_csv("temporal_structural_observed_vs_null_summary_v31.csv", index=False)
    overlap_all.to_csv("temporal_structural_null_overlap_gradient_v31.csv", index=False)
    compatibility_all.to_csv("temporal_structural_metric_null_compatibility_v31.csv", index=False)

    # --------------------------------------------------------
    # D. Community structure: 300 Infomap replicates/model/date
    # --------------------------------------------------------
    print("\n=== TEMPORAL INFOMAP COMMUNITY NULL ENSEMBLES ===")
    community_null_rows = []
    community_comparison_rows = []

    for di, date in enumerate(TEMPORAL_DATES):
        G = graphs[date]
        observed = observed_by_date[date]

        for mi, (name, (gen, kwargs)) in enumerate(generators.items()):
            model_seed = SEED + 5000000 + di * 100000 + mi * 10000
            print(f"{date} | {name}: {N_COMMUNITY_NULL} Infomap replicates...")
            cdf = run_community_ensemble(
                G, gen, n=N_COMMUNITY_NULL, seed=model_seed, **kwargs
            )
            keep = [
                "replicate","infomap_n_modules","infomap_codelength",
                "directed_modularity_Q_of_infomap_partition",
                "edge_overlap_fraction"
            ]
            out = cdf[keep].copy()
            out.insert(0, "null_model", name)
            out.insert(0, "date", date)
            community_null_rows.append(out)

            comp = compare_observed_to_null(
                observed, cdf[keep], name + "_community"
            )
            comp.insert(0, "date", date)
            community_comparison_rows.append(comp)

    community_null_all = pd.concat(community_null_rows, ignore_index=True)
    community_comparison_all = pd.concat(community_comparison_rows, ignore_index=True)
    community_null_all.to_csv("temporal_structural_community_null_ensembles_v31.csv", index=False)
    community_comparison_all.to_csv("temporal_structural_community_observed_vs_null_v31.csv", index=False)

    # --------------------------------------------------------
    # E. Date-specific funnel summary
    # Important: descriptive decision aid only. It is NOT interpreted as
    # "the first null where p>0.05 explains the metric."
    # --------------------------------------------------------
    funnel_rows = []
    for date in TEMPORAL_DATES:
        s = comparison_all[comparison_all["date"] == date].drop(columns="date")
        cs = community_comparison_all[
            community_comparison_all["date"] == date
        ].drop(columns="date")
        cp = compatibility_all[
            compatibility_all["date"] == date
        ].drop(columns="date")
        f = build_funnel_summary(s, cs, cp)
        f.insert(0, "date", date)
        funnel_rows.append(f)

    funnel_all = pd.concat(funnel_rows, ignore_index=True)
    funnel_all.to_csv("temporal_structural_null_funnel_summary_v31.csv", index=False)

    print("\nDONE.")
    print("Core outputs:")
    print(" - temporal_structural_null_feasibility_v31.csv")
    print(" - temporal_structural_N3_mixing_summary_v31.csv")
    print(" - temporal_structural_null_overlap_gradient_v31.csv")
    print(" - temporal_structural_observed_vs_null_summary_v31.csv")
    print(" - temporal_structural_community_observed_vs_null_v31.csv")
    print(" - temporal_structural_metric_null_compatibility_v31.csv")
    print(" - temporal_structural_null_funnel_summary_v31.csv")
    print("\nInterpret metric-by-metric across the null gradient; do not use the first non-significant null as an automatic explanatory threshold.")




In [ ]:
main()


## 4. Supplementary analysis: cumulative taxonomic food web

Construct the cumulative taxonomic network as a descriptive representation of the interaction architecture recorded across the monitoring programme. This network is not used for temporal inference.

In [ ]:
from pathlib import Path
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

DATA_DIR = Path(".")
NODES_FILE = DATA_DIR / "FW_lagoon_nodes_AUDITED.csv"
LINKS_FILE = DATA_DIR / "FW_lagoon_trophic_links_AUDITED.csv"

EXPECTED_NODES = 75
EXPECTED_LINKS = 394

In [ ]:
nodes = pd.read_csv(NODES_FILE)
links = pd.read_csv(LINKS_FILE)

print(f"Node table: {nodes.shape[0]} rows × {nodes.shape[1]} columns")
print(f"Trophic-link table: {links.shape[0]} rows × {links.shape[1]} columns")

display(nodes.head())
display(links.head())

In [ ]:
required_node_cols = {"node"}
required_link_cols = {"resource", "consumer"}

assert required_node_cols.issubset(nodes.columns)
assert required_link_cols.issubset(links.columns)
assert not nodes["node"].isna().any()
assert not nodes["node"].duplicated().any()
assert not links[["resource", "consumer"]].isna().any().any()
assert not links.duplicated(subset=["resource", "consumer"]).any()
assert not (links["resource"] == links["consumer"]).any()

node_set = set(nodes["node"])
endpoints = set(links["resource"]) | set(links["consumer"])
unknown_endpoints = sorted(endpoints - node_set)

assert unknown_endpoints == [], (
    f"Trophic-link endpoints absent from node table: {unknown_endpoints}"
)

print("Input validation passed.")

In [ ]:
G = nx.DiGraph()

# Add every audited node explicitly before adding trophic links.
metadata_cols = [c for c in nodes.columns if c != "node"]

for _, row in nodes.iterrows():
    attrs = {
        col: row[col]
        for col in metadata_cols
        if pd.notna(row[col])
    }
    G.add_node(row["node"], **attrs)

# Trophic direction: resource -> consumer.
G.add_edges_from(
    (row["resource"], row["consumer"])
    for _, row in links.iterrows()
)

assert G.number_of_nodes() == EXPECTED_NODES
assert G.number_of_edges() == EXPECTED_LINKS
assert nx.number_of_selfloops(G) == 0

print("Network construction passed.")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Links: {G.number_of_edges()}")

In [ ]:
S = G.number_of_nodes()
L = G.number_of_edges()

# Directed connectance when self-loops are excluded.
connectance = L / (S * (S - 1))

summary = pd.Series({
    "Nodes (S)": S,
    "Links (L)": L,
    "Connectance": connectance,
    "Isolates": nx.number_of_isolates(G),
    "Weakly connected components": nx.number_weakly_connected_components(G),
    "Strongly connected components": nx.number_strongly_connected_components(G),
    "Directed acyclic graph": nx.is_directed_acyclic_graph(G),
    "Nodes with in-degree = 0": sum(G.in_degree(n) == 0 for n in G.nodes),
    "Nodes with out-degree = 0": sum(G.out_degree(n) == 0 for n in G.nodes),
})

display(summary.to_frame("Value"))
print("Isolated nodes:", list(nx.isolates(G)))

In [ ]:
degree_table = pd.DataFrame({
    "node": list(G.nodes),
    "in_degree": [G.in_degree(n) for n in G.nodes],
    "out_degree": [G.out_degree(n) for n in G.nodes],
    "total_degree": [G.degree(n) for n in G.nodes],
})

degree_table = degree_table.merge(
    nodes[["node", "category", "FunctionalGroup"]],
    on="node",
    how="left",
).sort_values(["total_degree", "in_degree"], ascending=False)

display(degree_table)

In [ ]:
degree_table.sort_values("out_degree", ascending=False).head(10)

In [ ]:
pos = nx.spring_layout(G, seed=42, k=1.4)

plt.figure(figsize=(18, 16))
nx.draw_networkx_nodes(
    G, pos,
    node_size=260,
    edgecolors="black",
    linewidths=0.4,
)
nx.draw_networkx_edges(
    G, pos,
    arrows=True,
    arrowsize=8,
    width=0.6,
    alpha=0.45,
)
nx.draw_networkx_labels(
    G, pos,
    font_size=5.5,
)

plt.axis("off")
plt.tight_layout()
plt.show()